# Hilbert-Geometry Diagnostics
## Version 2: BoT-IoT | Day-23 final MAQT checkpoint

## Setup

In [1]:
import warnings

warnings.filterwarnings("ignore")

import json
from pathlib import Path

import pandas as pd
import pennylane as qp
import torch
from pennylane import numpy as np

In [2]:
from scripts.circuit import build_forward_circuit, create_quantum_device
from scripts.constants import DEFAULT_BATCH_SIZE, DEFAULT_NOISE_RATE, DEFAULT_SEED
from scripts.data import load_split, to_angles
from scripts.hilbert import hilbert_geometry_diagnostics, print_h1_report
from scripts.utils import get_torch_device

In [3]:
print(f"PyTorch version: {torch.__version__}")
print(f"PennyLane version: {qp.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.12.1+cu126
PennyLane version: 0.45.1
CUDA available: True


## Config

In [4]:
NOTEBOOK_NAME = "11.hilbert-geometry-diagnostics_v2-bot-iot"
DAY23_NOTEBOOK = "final-bot-iot-maqt-train"

dataset = "BoT-IoT"
target_col = "label_multiclass"
data_path = f"data/FROZEN/{dataset}"
frozen_feat_path = "team-artifacts/teamC_week3_FROZEN_handoff.json"

SEED = DEFAULT_SEED
MAX_PER_CLASS = 50  # H1 subsample per class (None = all)

## Load Team C's FROZEN Features

In [5]:
handoff = json.loads(Path(frozen_feat_path).read_text())
FEATURE_COLS = list(handoff["frozen_subsets"][dataset]["features"])
print(f"Team C selector: {handoff['frozen_subsets'][dataset]['selector']}")
print(f"k={len(FEATURE_COLS)} features: {FEATURE_COLS}")

Team C selector: QIGPSO
k=10 features: ['n_pkts_total', 'n_bytes_total', 'src_bytes', 'rate', 'pkt_size_min', 'pkt_size_max', 'pkt_size_mean', 'pkt_size_std', 'protocol', 'conn_state']


## Load Team A's FROZEN Train Split

Only the train split is needed for H1 fidelity-gap measurement against Day-23 prototypes (no EDA / balancing / retrain).

In [6]:
X_train, y_train, class_names, df_train = load_split(
    data_path,
    "train",
    target_col,
    categories=None,
    csv=True,
    selected_cols=FEATURE_COLS,
    return_df=True,
)

print(f"train: {X_train.shape}, y={y_train.shape}")
print(f"classes: {class_names}")
df_train.head(3)

train: (148729, 10), y=(148729,)
classes: ['DDoS', 'DoS', 'Normal', 'Reconnaissance']


,n_pkts_total,n_bytes_total,src_bytes,rate,pkt_size_min,pkt_size_max,pkt_size_mean,pkt_size_std,protocol,conn_state,label_multiclass
0,-0.262256,-0.798644,-0.774964,-0.356930,0.000000,0.123458,-0.266091,0.827772,0.0,0.0,DoS
1,0.427561,-0.121016,0.000000,-0.123704,0.000000,0.120431,0.030826,0.766490,0.0,0.0,DoS
2,0.768961,0.200956,0.368220,1.725276,0.761987,0.244690,0.422755,0.618673,0.0,0.0,DDoS


## Load Day-23 FROZEN Checkpoint

Reuse $\theta^\star$, prototypes, and the train-fitted scaler / PCA / angle bounds from Day-23.

In [7]:
LOAD_CHECKPOINT = True
ARTIFACTS_DIR = Path("final_notebooks") / "final_artifacts"
CHECKPOINT_CANDIDATES = [
    ARTIFACTS_DIR / f"{DAY23_NOTEBOOK}-checkpoint.pt",
]

batch_size = DEFAULT_BATCH_SIZE
seed = SEED

In [8]:
ckpt_path = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)

if not (LOAD_CHECKPOINT and ckpt_path is not None):
    raise FileNotFoundError(
        f"Day-23 checkpoint not found under {CHECKPOINT_CANDIDATES}. "
        f"Publish it from final_notebooks/{DAY23_NOTEBOOK}.ipynb first."
    )

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

theta_star = ckpt["theta"]
if not isinstance(theta_star, torch.nn.Parameter):
    theta_star = torch.nn.Parameter(theta_star)

prototypes = {int(k): v for k, v in ckpt["prototypes"].items()}
class_names = list(ckpt.get("class_names", class_names))
num_classes = int(ckpt.get("num_classes", len(class_names)))
num_qubits = int(ckpt["num_qubits"])
num_layers = int(ckpt.get("num_layers", 2))
noise_rate = float(ckpt.get("noise_rate", DEFAULT_NOISE_RATE))

scaler = ckpt["scaler"]
pca = ckpt.get("pca")
USE_PCA = bool(ckpt.get("use_pca", pca is not None))
x_min = np.asarray(ckpt["angle_x_min"])
x_max = np.asarray(ckpt["angle_x_max"])
angle_max = float(ckpt.get("angle_max", np.pi))
k95 = int(ckpt.get("k95", num_qubits))

print(f"loaded checkpoint: {ckpt_path}")
print(f"theta shape: {tuple(theta_star.shape)} | prototypes: {len(prototypes)}")
print(
    f"num_qubits={num_qubits} | num_layers={num_layers} | USE_PCA={USE_PCA} | k95={k95}"
)
print(
    f"feature_cols: {len(ckpt.get('feature_cols', FEATURE_COLS))} | classes: {class_names}"
)

loaded checkpoint: final_notebooks/final_artifacts/final-bot-iot-maqt-train-checkpoint.pt
theta shape: (2, 5, 3) | prototypes: 4
num_qubits=5 | num_layers=2 | USE_PCA=True | k95=5
feature_cols: 10 | classes: ['DDoS', 'DoS', 'Normal', 'Reconnaissance']


## Mapping Features $\rightarrow$ [0, $\pi$]

In [9]:
# raw -> scale -> optional pca -> [0, pi] using Day-23 train bounds
pca_for_angles = pca if USE_PCA else None
X_train_angles = to_angles(
    X_train, scaler, x_min, x_max, pca=pca_for_angles, angle_max=angle_max
)

print(f"angles train: {X_train_angles.shape} | USE_PCA={USE_PCA}")

angles train: (148729, 5) | USE_PCA=True


## Quantum Circuit: Angle Encoding, Data Reuploading, and Adding Noise

$x\xrightarrow[\text{encode}]{\phi(x)}\ket{\psi(x)}\xrightarrow[\text{variational}]{U(\theta)}\ket{\Phi(x)}\xrightarrow{\Lambda_p}\rho(x)$
- $x$: classical data
- $\phi(x)$: `qp.AngleEmbedding()`
- $\ket{\psi(x)}$: quantum state after encoding
- $U(\theta)$: `qp.StronglyEntanglingLayers()`
- $\ket{\Phi(x)}$: quantum state after variational transform
- $\rho(x)=\Lambda_p(\ket{\Phi(x)}\bra{\Phi(x)})$: standard depolarization channel to model NISQ noise

In [10]:
print(
    f"num_qubits={num_qubits} (from Day-23) | num_layers={num_layers} | noise_rate={noise_rate}"
)

# initialize devices
device = get_torch_device()
dev = create_quantum_device(num_qubits)

# define circuit
forward_circuit = build_forward_circuit(
    dev, num_qubits, num_layers, noise_rate=noise_rate
)

# move model tensors to device
theta_star = theta_star.to(device)
if not isinstance(theta_star, torch.nn.Parameter):
    theta_star = torch.nn.Parameter(theta_star)
prototypes = {int(k): v.to(device) for k, v in prototypes.items()}

print(f"device={device}")

num_qubits=5 (from Day-23) | num_layers=2 | noise_rate=0.01
device=cuda


## H1 Diagnostics

### Loss Functions
- **Intra-class loss** (infidelity to own prototype):
  $$
  L_{\mathrm{intra}}
  = \frac{1}{|\mathcal{C}|}
    \sum_{c \in \mathcal{C}}
    \mathbb{E}_{x \sim c}
    \big[1 - F(\rho(x), \rho_c)\big]
  $$
- **Inter-class loss** (negative mean prototype separation):
  $$
  L_{\mathrm{inter}}
  = - \frac{1}{|\mathcal{P}|}
    \sum_{(c,c') \in \mathcal{P}}
    D_{\mathrm{tr}}(\rho_c, \rho_{c'})
  $$
  where $\mathcal{P}$ is the set of unordered class pairs and $D_{\mathrm{tr}}$ is trace distance.

### Fidelity Gap
$$
\Delta F = \underbrace{\overline{F}_{\mathrm{intra}}}_{\mathrm{mean}_{c}\,(\mathrm{mean}_{x \in c} F(\rho(x), \rho_c))} - \underbrace{\overline{F}_{\mathrm{inter}}}_{\mathrm{mean}_{(c,c')} F(\rho_c,\rho_{c'})}
$$

where:

- $\overline{F}_{\mathrm{intra}} \approx 1 - L_{\mathrm{intra}}$ (proxy from the loss)
- $\overline{F}_{\mathrm{intra}} = \mathrm{mean\_intra\_fid} = \mathrm{mean}_{c}\,(\mathrm{mean}_{x \in c} F(\rho(x), \rho_c))$ (direct definition / H1 measurement)
    - $\mathrm{mean\_intra\_fid}_c = \mathrm{mean}_{x \in c} F(\rho(x), \rho_c)$ (per-class intra fidelity)
- $\overline{F}_{\mathrm{inter}} = \mathrm{mean\_inter\_fid} = \mathrm{mean}_{(c,c')} F(\rho_c,\rho_{c'})$
  (explicit fidelity; not from $L_{\mathrm{inter}}$)

### Ideal Trends
| Goal | Geometry | $L$ | Fidelity |
|---|---|---|---|
| Same class tighter | closer to $\rho_c$ | $L_{\mathrm{intra}} \downarrow$ | $F(\rho(x),\rho_c) \uparrow$ |
| Different classes farther | prototypes separate | $L_{\mathrm{inter}} \downarrow$ (more negative; $D_{\mathrm{tr}} \uparrow$) | $F(\rho_c,\rho_{c'}) \downarrow$ |
| Better Hilbert margin | - | - | $\Delta F \uparrow$ |

In [11]:
h1 = hilbert_geometry_diagnostics(
    theta_star,
    X_train_angles,
    y_train,
    prototypes,
    forward_circuit,
    class_names=class_names,
    device=device,
    max_per_class=MAX_PER_CLASS,
    seed=seed,
    batch_size=batch_size,
)
print_h1_report(h1)

=== H1 Hilbert geometry (fidelity gaps) ===
mean intra-class fidelity : 0.8229
mean inter-class fidelity : 0.7861
fidelity gap (intra-inter): 0.0368  ← want ↑
mean inter trace distance : 0.5088  ← want ↑

per-class intra fidelity:
  DDoS                         n=  50  F=0.8153
  DoS                          n=  50  F=0.8681
  Normal                       n=  50  F=0.7355
  Reconnaissance               n=  50  F=0.8727


In [12]:
# within-class (intra)
display(pd.DataFrame(h1["per_class"]).T)
display(pd.DataFrame(h1["pairs"]).sort_values("pair_inter_fid", ascending=False))

,n,mean_intra_fid_c,mean_intra_infidelity_c
DDoS,50.0,0.815318,0.184682
DoS,50.0,0.868113,0.131887
Normal,50.0,0.735515,0.264485
Reconnaissance,50.0,0.872746,0.127254


,pair,pair_inter_fid,pair_trace_distance
0,DDoS↔DoS,0.990056,0.109594
3,DoS↔Normal,0.816427,0.493233
1,DDoS↔Normal,0.811533,0.498121
5,Normal↔Reconnaissance,0.802878,0.534634
2,DDoS↔Reconnaissance,0.649506,0.707878
4,DoS↔Reconnaissance,0.646082,0.709288


## Logging

In [13]:
out_dir = Path("logs")
out_dir.mkdir(parents=True, exist_ok=True)
log_path = out_dir / f"{NOTEBOOK_NAME}.log"

lines = [
    f"notebook={NOTEBOOK_NAME}",
    f"checkpoint={ckpt_path}",
    f"dataset={dataset}",
    f"max_per_class={MAX_PER_CLASS}",
    f"mean_intra_fid={h1['mean_intra_fid']:.6f}",
    f"mean_inter_fid={h1['mean_inter_fid']:.6f}",
    f"fidelity_gap={h1['fidelity_gap']:.6f}",
    f"mean_inter_trace_distance={h1['mean_inter_trace_distance']:.6f}",
]
for name, row in h1["per_class"].items():
    lines.append(f"intra[{name}]: n={row['n']} F={row['mean_intra_fid_c']:.6f}")
for row in h1["pairs"]:
    lines.append(
        f"pair[{row['pair']}]: F={row['pair_inter_fid']:.6f} TD={row['pair_trace_distance']:.6f}"
    )

log_path.write_text("\n".join(lines) + "\n")
print(f"wrote {log_path}")

wrote logs/11.hilbert-geometry-diagnostics_v2-bot-iot.log
